# Log-Likelihood and Maximum Likelihood

This notebook explains why every statistician, every machine-learning library and every fitting routine works with the **log-likelihood** $\sum \ln p(x_i)$ instead of the likelihood $\prod p(x_i)$. The logarithm does two jobs at once: it rescues the computation from numerical underflow, and it turns a hopeless derivative of a product of a thousand factors into a sum of a thousand easy terms — without moving the maximum.

This is notebook 03 of module 05 (**Information and probability**) in the **Learning Logarithms** series.

## 1. Introduction

A coin is tossed $1000$ times and lands heads $620$ times. Which bias $p$ best explains the data? **Maximum likelihood** says: choose the $p$ that makes the observed data most probable. For independent tosses the probability of the observed sequence is a product of $1000$ factors,

$$
L(p) = \prod_{i=1}^{1000} p^{x_i} (1 - p)^{1 - x_i} = p^{620} (1 - p)^{380},
$$

and we want the $p$ that maximizes it. Two problems appear immediately:

1. **The number is absurdly small.** At the best value $p = 0.62$, $L(p) \approx 10^{-288}$. The smallest positive normal double is about $2.2 \times 10^{-308}$; with $2000$ tosses, or with the probabilities of a few thousand data points from any realistic model, the product **underflows to exactly $0$** and every candidate $p$ looks equally bad.
2. **The derivative is a mess.** Differentiating a product of $n$ factors by the product rule gives $n$ terms, each itself a product of $n - 1$ factors. For a model with several parameters and $n = 10^6$ data points this is unworkable.

Both problems disappear with one move. Take $\ln$:

$$
\ell(p) = \ln L(p) = \sum_{i=1}^{n} \ln p(x_i) = 620 \ln p + 380 \ln(1 - p).
$$

The product becomes a sum (Napier's trick again), the tiny number $10^{-288}$ becomes the harmless $-664$, and because $\ln$ is strictly increasing, **the $p$ that maximizes $\ell$ is the same $p$ that maximizes $L$**.

**Prerequisites**

- The laws of logarithms, $\ln(xy) = \ln x + \ln y$ and $\ln x^k = k \ln x$ — [The laws of logarithms](../00_foundations/03_laws_of_logarithms.ipynb).
- Derivative of $\ln x$ — [Derivatives of ln and exp](../01_the_natural_logarithm/03_derivatives_of_ln_and_exp.ipynb), and [Derivative of logarithmic functions](../../../LearningCalculus/hypersklill/02_derivaties/05_derivative_of_logarithmic_functions.ipynb) (LearningCalculus).
- What maximum likelihood is — [Intro to maximum likelihood](../../../LearningProbability/coursera/05_point_estimation/01_intro_to_maximum_likelihood.ipynb) (LearningProbability).
- Floating-point range and underflow — [Floating point and numerical stability](../03_computing_logarithms/04_floating_point_and_numerical_stability.ipynb).
- Entropy, for the last remark of section 2 — [Entropy and information](./02_entropy_and_information.ipynb).

After completing this notebook you will be able to:

- Explain why maximizing $\ln L$ gives the same answer as maximizing $L$.
- Demonstrate numerical underflow of a likelihood and avoid it with log-probabilities.
- Derive the maximum-likelihood estimates for the Bernoulli and Gaussian models by differentiating the log-likelihood.
- Read likelihood and log-likelihood plots and explain why the log version is concave and easy to optimize.
- Connect the average log-likelihood to the cross-entropy of notebook 04.

## 2. Theory

### 2.1 A monotone map does not move the maximum

If $g$ is strictly increasing, then $f(a) > f(b) \iff g(f(a)) > g(f(b))$. Every comparison between candidate parameters comes out the same, so

$$
\arg\max_\theta L(\theta) = \arg\max_\theta \, \ln L(\theta) \qquad (L(\theta) > 0).
$$

The **value** at the maximum changes; the **location** does not. The logarithm is the right monotone map because of what it does to products:

| Exponent form (likelihood) | Log form (log-likelihood) |
|---|---|
| $L(\theta) = \prod_{i=1}^n p(x_i \mid \theta)$ | $\ell(\theta) = \sum_{i=1}^n \ln p(x_i \mid \theta)$ |
| $L = e^{\ell}$ | $\ell = \ln L$ |
| derivative: product rule, $n$ terms of $n$ factors | derivative: $\sum_i \dfrac{\partial}{\partial \theta} \ln p(x_i \mid \theta)$ |

### 2.2 The numerical picture

A double-precision float stores an exponent between about $-1022$ and $1023$ (powers of $2$), so the smallest positive normal value is $2^{-1022} \approx 2.2 \times 10^{-308}$ (subnormals reach down to $5 \times 10^{-324}$ with lost precision). A product of $n$ probabilities that are each around $p$ has size $p^n = 10^{n \log_{10} p}$; it underflows when

$$
n \log_{10} p < -308 \quad\Longleftrightarrow\quad n > \frac{308}{-\log_{10} p}.
$$

For $p = 0.5$ that is $n > 1023$: about a thousand coin flips. The log-likelihood $n \ln p$ is only about $-700$ there — no problem at all for a float. Working on the log scale moves the number from the *exponent* of the float into its *mantissa*.

### 2.3 Bernoulli: the MLE is the sample proportion

Data: $n$ tosses, $k$ heads. Log-likelihood:

$$
\ell(p) = k \ln p + (n - k) \ln(1 - p), \qquad 0 < p < 1.
$$

Differentiate (using $\frac{d}{dp} \ln p = \frac1p$) and set to zero:

$$
\ell'(p) = \frac{k}{p} - \frac{n - k}{1 - p} = 0
\;\Longrightarrow\; k(1 - p) = (n - k) p
\;\Longrightarrow\; \hat p = \frac{k}{n}.
$$

The second derivative $\ell''(p) = -\frac{k}{p^2} - \frac{n - k}{(1 - p)^2} < 0$, so $\ell$ is **concave**: a single peak, no local traps. For $620$ heads in $1000$ tosses, $\hat p = 0.62$.

### 2.4 Gaussian: the MLEs are the sample mean and the (biased) sample variance

Data $x_1, \dots, x_n$, model $\mathcal N(\mu, \sigma^2)$ with density $\frac{1}{\sigma\sqrt{2\pi}} e^{-(x - \mu)^2 / (2\sigma^2)}$. The logarithm undoes the exponential in every factor:

$$
\ell(\mu, \sigma) = \sum_{i=1}^n \left[-\ln \sigma - \tfrac12 \ln(2\pi) - \frac{(x_i - \mu)^2}{2\sigma^2}\right]
= -n \ln \sigma - \frac{n}{2}\ln(2\pi) - \frac{1}{2\sigma^2}\sum_{i=1}^n (x_i - \mu)^2 .
$$

- $\frac{\partial \ell}{\partial \mu} = \frac{1}{\sigma^2}\sum (x_i - \mu) = 0 \;\Longrightarrow\; \hat\mu = \bar x = \frac1n \sum x_i$.
- $\frac{\partial \ell}{\partial \sigma} = -\frac{n}{\sigma} + \frac{1}{\sigma^3}\sum (x_i - \mu)^2 = 0 \;\Longrightarrow\; \hat\sigma^2 = \frac1n \sum (x_i - \bar x)^2$.

Maximizing the Gaussian log-likelihood in $\mu$ is the same as minimizing $\sum (x_i - \mu)^2$: **least squares is maximum likelihood with Gaussian noise**, and the log is what reveals it.

### 2.5 Average log-likelihood is a (negative) cross-entropy

Divide by $n$. By the law of large numbers, if the data really come from a distribution $p_{\text{true}}$,

$$
\frac{1}{n}\ell(\theta) = \frac1n \sum_{i=1}^n \ln q_\theta(x_i) \;\longrightarrow\; E_{p_{\text{true}}}[\ln q_\theta(X)] = -H(p_{\text{true}}, q_\theta),
$$

minus the **cross-entropy** (in nats) between the truth and the model. So maximum likelihood is "choose the model whose surprise on the data is smallest on average" — the bridge from notebook 02 to notebook 04.

## 3. Worked Examples

### Example 1 — how small is the likelihood?

$620$ heads in $1000$ tosses, at $p = 0.62$:

$$
\log_{10} L = 620 \log_{10} 0.62 + 380 \log_{10} 0.38 \approx 620(-0.20761) + 380(-0.42022) = -128.72 - 159.68 = -288.40 .
$$

So $L \approx 10^{-288.4} \approx 4 \times 10^{-289}$: representable, but only just. Doubling the data to $1240$ heads in $2000$ tosses squares $L$ and gives about $10^{-577}$, which underflows to $0$.

### Example 2 — Bernoulli MLE for small data

$3$ heads in $10$ tosses: $\ell(p) = 3 \ln p + 7 \ln(1 - p)$, $\ell'(p) = \frac{3}{p} - \frac{7}{1 - p} = 0 \Rightarrow 3 - 3p = 7p \Rightarrow \hat p = 0.3$.

Values: $\ell(0.3) = 3 \ln 0.3 + 7 \ln 0.7 \approx -3.612 - 2.497 = -6.109$, while $\ell(0.5) = 10 \ln 0.5 \approx -6.931$. The likelihood ratio is $e^{-6.109 + 6.931} = e^{0.822} \approx 2.28$: the data are about $2.3$ times more probable under $p = 0.3$ than under a fair coin.

### Example 3 — Gaussian MLE

Data $2, 4, 4, 5, 7, 8$. $\hat\mu = 30 / 6 = 5$. Squared deviations: $9, 1, 1, 0, 4, 9$, sum $24$, so $\hat\sigma^2 = 24 / 6 = 4$ and $\hat\sigma = 2$. (The unbiased sample variance divides by $n - 1 = 5$ instead: $4.8$.)

### Example 4 — differences of log-likelihoods are log-ratios

Comparing two models on the same data needs only $\ell_1 - \ell_2 = \ln(L_1 / L_2)$. On the log scale the comparison is a subtraction, even when $L_1$ and $L_2$ are both too small to store. For $620/1000$, $\ell(0.62) - \ell(0.5) \approx -664.1 - (-693.1) = 29.0$, so $L(0.62)/L(0.5) = e^{29.0} \approx 4 \times 10^{12}$.

## 4. Python Examples

### 4.1 Watching the likelihood underflow

Multiply the probabilities of $n$ fair-coin outcomes one at a time and see when the product becomes $0$; keep the sum of logs alongside.

In [ ]:
import math
import sys

print("smallest positive normal float:", sys.float_info.min)
print("smallest subnormal float      :", 5e-324)
print()

for n_tosses in [100, 500, 1000, 1074, 1075, 2000]:
    product = 1.0
    log_sum = 0.0
    for _ in range(n_tosses):
        product *= 0.5
        log_sum += math.log(0.5)
    print(f"n = {n_tosses:>5}: product = {product:<12.4g} sum of ln = {log_sum:10.3f}   (log10 of the true product = {log_sum / math.log(10):8.2f})")

The product reaches the subnormal range after $1023$ halvings and becomes exactly $0$ after $1075$. The sum of logarithms goes on happily: it just keeps getting more negative, one $\ln 0.5 \approx -0.693$ at a time.

### 4.2 The likelihood of real-looking data, both ways

A sample of $1000$ points from a normal distribution: the product of the densities is $0$ for every candidate mean, but the log-likelihood picks out the right one.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
data = rng.normal(loc=3.0, scale=2.0, size=1000)


def gaussian_density(x, mu, sigma):
    return np.exp(-((x - mu) ** 2) / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))


def gaussian_log_density(x, mu, sigma):
    return -np.log(sigma) - 0.5 * np.log(2 * np.pi) - (x - mu) ** 2 / (2 * sigma**2)


for mu in [2.0, 3.0, 4.0]:
    likelihood = np.prod(gaussian_density(data, mu, 2.0))
    log_likelihood = np.sum(gaussian_log_density(data, mu, 2.0))
    print(f"mu = {mu}: product of densities = {likelihood},  log-likelihood = {log_likelihood:.2f}")

### 4.3 Deriving the MLEs symbolically

SymPy differentiates the log-likelihoods of sections 2.3 and 2.4 and solves for the critical points.

In [ ]:
import sympy as sp

p, n, k = sp.symbols("p n k", positive=True)
bernoulli_log_likelihood = k * sp.log(p) + (n - k) * sp.log(1 - p)
score = sp.diff(bernoulli_log_likelihood, p)
print("Bernoulli: l'(p) =", sp.simplify(score))
print("           p_hat =", sp.solve(sp.Eq(score, 0), p))
print("           l''(p) =", sp.simplify(sp.diff(bernoulli_log_likelihood, p, 2)))

mu = sp.symbols("mu", real=True)
sigma = sp.symbols("sigma", positive=True)
xs = sp.symbols("x1:5", real=True)           # four symbolic data points
gaussian_log_likelihood = sum(-sp.log(sigma) - sp.log(2 * sp.pi) / 2 - (x - mu) ** 2 / (2 * sigma**2) for x in xs)
mu_hat = sp.solve(sp.diff(gaussian_log_likelihood, mu), mu)[0]
sigma_squared_hat = sp.solve(sp.diff(gaussian_log_likelihood, sigma).subs(mu, mu_hat), sigma)[0] ** 2
print()
print("Gaussian:  mu_hat =", mu_hat)
x_bar = sum(xs) / len(xs)
biased_sample_variance = sum((x - x_bar) ** 2 for x in xs) / len(xs)
print("           sigma_hat^2 - (1/n) sum (x_i - x_bar)^2 =", sp.simplify(sigma_squared_hat - biased_sample_variance))

### 4.4 Numerical MLE and the size of the gradients

On a grid, both $L$ and $\ell$ peak at $\hat p = 0.62$ for the $620/1000$ data. But the **gradients** differ enormously: an optimizer following $L'(p)$ takes steps of size around $10^{-290}$, while $\ell'(p)$ is a sensible number. This is the second reason every optimizer works with $\ell$.

In [ ]:
import numpy as np

heads, tosses = 620, 1000
grid = np.linspace(0.40, 0.80, 4001)
log_likelihood = heads * np.log(grid) + (tosses - heads) * np.log(1 - grid)
likelihood = np.exp(log_likelihood)

print("argmax of likelihood     :", round(grid[np.argmax(likelihood)], 4))
print("argmax of log-likelihood :", round(grid[np.argmax(log_likelihood)], 4))
print("max likelihood value     :", likelihood.max())
print("max log-likelihood value :", round(log_likelihood.max(), 3))

for p in [0.5, 0.6]:
    log_likelihood_gradient = heads / p - (tosses - heads) / (1 - p)
    likelihood_at_p = np.exp(heads * np.log(p) + (tosses - heads) * np.log(1 - p))
    likelihood_gradient = likelihood_at_p * log_likelihood_gradient       # L' = L * l'
    print(f"at p = {p}: dl/dp = {log_likelihood_gradient:9.2f},   dL/dp = {likelihood_gradient:.3e}")

The relation $L' = L \cdot \ell'$ (the derivative of $\ln L$ is $L'/L$) shows why: the likelihood gradient carries the tiny factor $L$ in front.

### 4.5 Gradient ascent on the log-likelihood

Fitting the Gaussian with plain gradient ascent on $\ell(\mu, \ln\sigma)$, averaged over the data. Optimizing $\ln \sigma$ instead of $\sigma$ keeps $\sigma$ positive automatically — another everyday use of the logarithm.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
data = rng.normal(loc=3.0, scale=2.0, size=1000)

mu, log_sigma = 0.0, 0.0
learning_rate = 0.1
for step in range(500):
    sigma = np.exp(log_sigma)
    residuals = data - mu
    gradient_mu = np.mean(residuals) / sigma**2
    gradient_log_sigma = -1 + np.mean(residuals**2) / sigma**2      # d/d(ln sigma) of the average log-likelihood
    mu += learning_rate * gradient_mu
    log_sigma += learning_rate * gradient_log_sigma

print(f"gradient ascent: mu = {mu:.4f}, sigma = {np.exp(log_sigma):.4f}")
print(f"closed form    : mu = {data.mean():.4f}, sigma = {data.std():.4f}   (np.std divides by n: the MLE)")

### 4.6 Comparing models when both likelihoods are zero

For the Gaussian sample of 4.2, the products of densities were all `0.0`, so "which mean is better?" cannot be answered on the likelihood scale. On the log scale it is a subtraction, and $e^{\ell_1 - \ell_2}$ is the likelihood ratio — a perfectly ordinary number even though neither likelihood can be stored.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
data = rng.normal(loc=3.0, scale=2.0, size=1000)


def gaussian_log_likelihood(mu, sigma):
    return np.sum(-np.log(sigma) - 0.5 * np.log(2 * np.pi) - (data - mu) ** 2 / (2 * sigma**2))


log_likelihood_3 = gaussian_log_likelihood(3.0, 2.0)
for mu in [2.0, 2.9, 3.1, 4.0]:
    difference = log_likelihood_3 - gaussian_log_likelihood(mu, 2.0)
    print(f"l(mu=3) - l(mu={mu}) = {difference:8.3f}   ->  L(3)/L({mu}) = e^{difference:.2f} = {np.exp(difference):.3g}")

Note that $\mu = 2.9$ beats the true value $\mu = 3$ slightly (ratio below $1$): the sample mean is $2.942$, and the data cannot know the truth — they only know which model they make least surprising. Moving a whole unit away from $\bar x$ costs a factor of $10^{48}$ or more.

## 5. Visualizations

### 5.1 Likelihood versus log-likelihood as the data grow

Left: the likelihood for $10$, $100$ and $1000$ tosses (each with $62\%$ heads), divided by its maximum so the three curves fit on one axis — without that rescaling the $n = 1000$ curve would be invisible next to the others. Right: the log-likelihood minus its maximum. All curves peak at $0.62$; the log versions are smooth, concave and on a readable scale.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

grid = np.linspace(0.3, 0.9, 1000)
fig, (ax_likelihood, ax_log) = plt.subplots(1, 2, figsize=(12, 4.5))
for tosses, colour in [(10, "tab:blue"), (100, "tab:orange"), (1000, "tab:green")]:
    heads = round(0.62 * tosses)
    log_likelihood = heads * np.log(grid) + (tosses - heads) * np.log(1 - grid)
    peak = log_likelihood.max()
    ax_likelihood.plot(grid, np.exp(log_likelihood - peak), color=colour, label=f"n = {tosses}  (max L = 10^{peak / np.log(10):.1f})")
    ax_log.plot(grid, log_likelihood - peak, color=colour, label=f"n = {tosses}")

for ax in (ax_likelihood, ax_log):
    ax.axvline(0.62, color="gray", linestyle=":")
    ax.set_xlabel("p")
    ax.grid(True, alpha=0.3)
ax_likelihood.set_ylabel("L(p) / max L")
ax_likelihood.set_title("Likelihood (rescaled to peak 1)")
ax_likelihood.legend(loc="upper left", fontsize=9)
ax_log.set_ylabel("l(p) - max l")
ax_log.set_ylim(-20, 1)
ax_log.set_title("Log-likelihood: concave, same argmax")
ax_log.legend(loc="lower center")
plt.tight_layout()
plt.show()

More data make the peak **sharper**: the log-likelihood curvature at the maximum is $-\ell''(\hat p) = n / (\hat p (1 - \hat p))$, growing linearly in $n$. Near its peak every log-likelihood looks like a downward parabola — which is why the likelihood itself looks like a bell curve.

### 5.2 The underflow cliff on a log axis

For products of $n$ probabilities equal to $0.5$, $0.1$ and $0.01$, the computed product (left) falls off a cliff to $0$ at the underflow limit, while the log-likelihood (right) is a straight line that never fails. The $y$ axis on the left is logarithmic, so exponential decay appears as a straight line until the float format gives out.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_values = np.arange(1, 1601)
fig, (ax_product, ax_log) = plt.subplots(1, 2, figsize=(12, 4.5))
for probability, colour in [(0.5, "tab:blue"), (0.1, "tab:orange"), (0.01, "tab:green")]:
    products = np.cumprod(np.full(n_values.size, probability))
    shown = np.where(products > 0, products, np.nan)          # zeros cannot be drawn on a log axis
    underflow_n = n_values[np.argmax(products == 0)] if np.any(products == 0) else None
    ax_product.plot(n_values, shown, color=colour, label=f"p = {probability}, becomes 0 at n = {underflow_n}")
    ax_log.plot(n_values, n_values * np.log(probability), color=colour, label=f"n ln({probability})")

ax_product.set_yscale("log")
ax_product.axhline(2.2e-308, color="crimson", linestyle="--", linewidth=1)
ax_product.text(380, 1e-318, "smallest normal float, 2.2e-308", color="crimson", fontsize=9)
ax_product.set_ylim(1e-330, 1e10)
ax_product.set_yticks([10.0**e for e in range(0, -301, -50)])
ax_product.set_xlabel("number of factors n")
ax_product.set_ylabel("computed product p^n (log scale)")
ax_product.set_title("The product underflows")
ax_product.grid(True, which="both", alpha=0.3)
ax_product.legend(loc="upper right", fontsize=9)

ax_log.set_xlabel("number of factors n")
ax_log.set_ylabel("sum of ln p (linear scale)")
ax_log.set_title("The sum of logs does not")
ax_log.grid(True, alpha=0.3)
ax_log.legend(loc="lower left")
plt.tight_layout()
plt.show()

### 5.3 The Gaussian log-likelihood surface

Contours of $\ell(\mu, \sigma)$ for the sample of section 4.2. The surface has a single peak at $(\bar x, \hat\sigma)$ — the closed-form MLE.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
data = rng.normal(loc=3.0, scale=2.0, size=1000)

mu_grid = np.linspace(2.5, 3.5, 200)
sigma_grid = np.linspace(1.7, 2.4, 200)
mu_mesh, sigma_mesh = np.meshgrid(mu_grid, sigma_grid)
n = data.size
sum_x = data.sum()
sum_x2 = (data**2).sum()
sum_squared_residuals = sum_x2 - 2 * mu_mesh * sum_x + n * mu_mesh**2
log_likelihood = -n * np.log(sigma_mesh) - 0.5 * n * np.log(2 * np.pi) - sum_squared_residuals / (2 * sigma_mesh**2)

fig, ax = plt.subplots(figsize=(8, 6))
levels = log_likelihood.max() - np.array([20, 10, 5, 2, 0.5])
contour = ax.contour(mu_mesh, sigma_mesh, log_likelihood, levels=levels, cmap="viridis")
ax.clabel(contour, fmt=lambda value: f"max - {log_likelihood.max() - value:.1f}", fontsize=8)
ax.plot(data.mean(), data.std(), "r*", markersize=14, label=f"MLE ({data.mean():.3f}, {data.std():.3f})")
ax.plot(3.0, 2.0, "kx", markersize=10, label="true (3, 2)")
ax.set_xlabel("mu")
ax.set_ylabel("sigma")
ax.set_title("Gaussian log-likelihood contours")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.show()

## 6. Exercises

### Basic

1. A coin shows $7$ heads in $10$ tosses. Write $\ell(p)$, differentiate, and find $\hat p$.
2. Let $f(\theta) = \theta^2 e^{-\theta}$ for $\theta > 0$. Find the maximizer of $f$ by maximizing $\ln f$ instead. Why is that easier?
3. Compute $\ln(0.5^{1000})$ and $\log_{10}(0.5^{1000})$ without multiplying anything out.

### Intermediate

4. **Estimate it first, then compute.** A model assigns probability about $0.2$ to each of $500$ independent observations. Roughly what power of $10$ is the likelihood? Will a double hold it? Check with Python.
5. Counts of emails per hour: $2, 4, 3, 0, 5, 1, 3, 2$. For the Poisson model $p(x \mid \lambda) = e^{-\lambda}\lambda^x / x!$, write $\ell(\lambda)$ and show $\hat\lambda = \bar x$. Compute it.
6. Waiting times $0.5, 1.2, 0.3, 2.0, 0.8, 1.7$ minutes, modelled as exponential with density $\lambda e^{-\lambda x}$. Find $\hat\lambda$.

### Advanced

7. For the Gaussian, show with SymPy (or by hand) that $\partial \ell / \partial \sigma = 0$ gives $\hat\sigma^2 = \frac1n\sum(x_i - \bar x)^2$, and use a simulation of $20{,}000$ samples of size $n = 5$ from $\mathcal N(0, 1)$ to show that its average is about $\frac{n-1}{n} = 0.8$, not $1$.
8. Simulate $n = 100{,}000$ tosses with true $p = 0.3$. For $q \in \{0.2, 0.3, 0.5\}$ compare $\frac1n \ell(q)$ with $-H(p, q) = p \ln q + (1 - p)\ln(1 - q)$.
9. The curvature of the log-likelihood measures precision: the standard error of $\hat p$ is approximately $1 / \sqrt{-\ell''(\hat p)}$. Show that for the Bernoulli model this equals $\sqrt{\hat p (1 - \hat p) / n}$ and evaluate it for $620$ heads in $1000$ tosses.

## 7. Solutions

1. $\ell(p) = 7 \ln p + 3 \ln(1 - p)$, $\ell'(p) = \frac7p - \frac{3}{1 - p} = 0 \Rightarrow 7 - 7p = 3p \Rightarrow \hat p = 0.7$.
2. $\ln f = 2 \ln \theta - \theta$, derivative $\frac{2}{\theta} - 1 = 0 \Rightarrow \hat\theta = 2$. No product rule is needed: the log split $\theta^2 e^{-\theta}$ into a sum of two simple terms.
3. $\ln(0.5^{1000}) = 1000 \ln 0.5 \approx -693.15$; $\log_{10}(0.5^{1000}) = -1000 \log_{10} 2 \approx -301.03$, i.e. $0.5^{1000} \approx 10^{-301}$ — just above the normal float limit.
4. $\log_{10} 0.2 \approx -0.7$, so the likelihood is about $10^{-350}$, below even the subnormal limit $5 \times 10^{-324}$: it underflows to $0$. The log-likelihood $500 \ln 0.2 \approx -804.7$ is fine.
5. $\ell(\lambda) = \sum (-\lambda + x_i \ln \lambda - \ln x_i!) = -n\lambda + (\sum x_i) \ln \lambda - \text{const}$. $\ell'(\lambda) = -n + \sum x_i / \lambda = 0 \Rightarrow \hat\lambda = \bar x = 20 / 8 = 2.5$ emails per hour.
6. $\ell(\lambda) = n \ln \lambda - \lambda \sum x_i$, $\ell' = n/\lambda - \sum x_i = 0 \Rightarrow \hat\lambda = n / \sum x_i = 6 / 6.5 \approx 0.923$ per minute.
7. Section 2.4 gives $\hat\sigma^2 = \frac1n \sum (x_i - \bar x)^2$. Its expectation is $\frac{n-1}{n}\sigma^2$, because $\bar x$ is fitted to the same data and sits closer to the points than the true mean. The simulation below gives about $0.8$ for $n = 5$.
8. The averages agree to about two or three decimals, and the largest of them is at $q = 0.3$: on average, the true model is the least surprised by its own data (notebook 04 calls the gap the KL divergence).
9. $-\ell''(\hat p) = \frac{k}{\hat p^2} + \frac{n - k}{(1 - \hat p)^2} = \frac{n}{\hat p} + \frac{n}{1 - \hat p} = \frac{n}{\hat p(1 - \hat p)}$ using $k = n\hat p$. So the standard error is $\sqrt{\hat p(1 - \hat p)/n} = \sqrt{0.62 \times 0.38 / 1000} \approx 0.0153$.

Numerical checks:

In [ ]:
import math

import numpy as np
import sympy as sp

p = sp.symbols("p", positive=True)
print("Ex 1:", sp.solve(sp.diff(7 * sp.log(p) + 3 * sp.log(1 - p), p), p))
theta = sp.symbols("theta", positive=True)
print("Ex 2:", sp.solve(sp.diff(sp.log(theta**2 * sp.exp(-theta)), theta), theta))
print("Ex 3:", round(1000 * math.log(0.5), 2), round(1000 * math.log10(0.5), 2))
print("Ex 4: log10 L =", round(500 * math.log10(0.2), 1), " product =", 0.2**500, " log-likelihood =", round(500 * math.log(0.2), 1))

counts = np.array([2, 4, 3, 0, 5, 1, 3, 2])
lam = sp.symbols("lambda", positive=True)
poisson_log_likelihood = sum(-lam + int(x) * sp.log(lam) - sp.log(sp.factorial(int(x))) for x in counts)
print("Ex 5:", sp.solve(sp.diff(poisson_log_likelihood, lam), lam), " mean =", counts.mean())

waits = np.array([0.5, 1.2, 0.3, 2.0, 0.8, 1.7])
print("Ex 6: lambda_hat =", round(len(waits) / waits.sum(), 4))

rng = np.random.default_rng(7)
samples = rng.normal(0, 1, size=(20_000, 5))
mle_variances = samples.var(axis=1)                 # divides by n
print("Ex 7: mean of sigma_hat^2 =", round(mle_variances.mean(), 4), " (n-1)/n =", 4 / 5)

true_p = 0.3
tosses = rng.random(100_000) < true_p
heads = tosses.sum()
for q in [0.2, 0.3, 0.5]:
    average_log_likelihood = (heads * math.log(q) + (tosses.size - heads) * math.log(1 - q)) / tosses.size
    minus_cross_entropy = true_p * math.log(q) + (1 - true_p) * math.log(1 - q)
    print(f"Ex 8: q = {q}: (1/n) l = {average_log_likelihood:.4f},  -H(p, q) = {minus_cross_entropy:.4f}")

print("Ex 9: standard error =", round(math.sqrt(0.62 * 0.38 / 1000), 5))

## 8. Conclusion

- The likelihood of independent data is a **product**; its logarithm is a **sum**: $\;L(\theta) = \prod p(x_i \mid \theta) \iff \ell(\theta) = \sum \ln p(x_i \mid \theta)$.
- Because $\ln$ is strictly increasing, $\arg\max L = \arg\max \ell$: taking logs changes the height of the peak, never its location.
- Numerically, products of a few thousand probabilities underflow to $0$; the log-likelihood stays a modest negative number. Comparisons become differences, $\ell_1 - \ell_2 = \ln(L_1/L_2)$.
- Analytically, the derivative of a sum is a sum. The Bernoulli MLE is $\hat p = k/n$; the Gaussian MLEs are $\bar x$ and $\frac1n\sum(x_i - \bar x)^2$; Gaussian maximum likelihood is least squares.
- The log-likelihood is concave for these models and its curvature measures how precisely the data pin the parameter down; its average is minus the cross-entropy between the data and the model.
- The picture: likelihood curves become unreadably narrow and tiny as $n$ grows; log-likelihood curves are smooth downward parabolas on a sensible scale.

**Next:** notebook 04 turns the average log-likelihood into a loss function: the **cross-entropy** $H(p, q)$, the **KL divergence** that measures how far a predicted distribution is from the truth, and the **softmax** that produces those predictions in every classifier.